# MWE 02 — Optional OpenPNM solver cross-check

This compares `voids` and OpenPNM `StokesFlow` using the *same* throat hydraulic conductance values.
The notebook now includes both:
- a constant-viscosity reference case
- a pressure-coupled water-viscosity case using the new `thermo` backend with cached interpolation

Because thermodynamic backends interpret pressure as **absolute pressure in Pa**, the coupled case
uses `pin=2e5 Pa`, `pout=1e5 Pa`. The older gauge-only choice `pin=1`, `pout=0` remains fine for
constant-viscosity checks but is not physically meaningful for `mu(P, T)`.
Run in `pixi run -e test python -m jupyter lab`.


In [1]:
import numpy as np

from voids.examples import make_linear_chain_network
from voids.physics.singlephase import FluidSinglePhase, PressureBC
from voids.physics.thermo import TabulatedWaterViscosityModel
from voids.benchmarks.crosscheck import (
    crosscheck_singlephase_roundtrip_openpnm_dict,
    crosscheck_singlephase_with_openpnm,
)

In [2]:
net_constant = make_linear_chain_network()
fluid_constant = FluidSinglePhase(viscosity=1.0)
bc_constant = PressureBC("inlet_xmin", "outlet_xmax", pin=1.0, pout=0.0)

net_thermo = make_linear_chain_network()
net_thermo.throat.pop("hydraulic_conductance")
net_thermo.throat["area"] = np.sqrt(8.0 * np.pi) * np.ones(net_thermo.Nt)
fluid_thermo = FluidSinglePhase(
    viscosity=1.0e-3,
    viscosity_model=TabulatedWaterViscosityModel.from_backend(
        "thermo",
        temperature=298.15,
        pressure_points=128,
    ),
)
bc_thermo = PressureBC("inlet_xmin", "outlet_xmax", pin=2.0e5, pout=1.0e5)

In [3]:
print("Constant-viscosity dictionary round-trip:")
print(
    crosscheck_singlephase_roundtrip_openpnm_dict(
        net_constant, fluid_constant, bc_constant, axis="x"
    )
)

Constant-viscosity dictionary round-trip:
SinglePhaseCrosscheckSummary(reference='openpnm_dict_roundtrip', axis='x', permeability_abs_diff=0.0, permeability_rel_diff=0.0, total_flow_abs_diff=0.0, total_flow_rel_diff=0.0, details={'k_voids': 0.5, 'k_ref': 0.5, 'Q_voids': 0.5, 'Q_ref': 0.5})


In [4]:
try:
    print("\nConstant-viscosity OpenPNM cross-check:")
    s_constant = crosscheck_singlephase_with_openpnm(
        net_constant,
        fluid_constant,
        bc_constant,
        axis="x",
    )
    print(s_constant)
    print(s_constant.details)

    print("\nPressure-coupled thermo/OpenPNM cross-check:")
    s_thermo = crosscheck_singlephase_with_openpnm(
        net_thermo,
        fluid_thermo,
        bc_thermo,
        axis="x",
    )
    print(s_thermo)
    print(s_thermo.details)
except ImportError as exc:
    print(exc)


Constant-viscosity OpenPNM cross-check:


/Users/dtvolpatto/Work/voids/.pixi/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SinglePhaseCrosscheckSummary(reference='openpnm_stokesflow', axis='x', permeability_abs_diff=0.0, permeability_rel_diff=0.0, total_flow_abs_diff=0.0, total_flow_rel_diff=0.0, details={'k_voids': 0.5, 'k_ref': 0.5, 'Q_voids': 0.5, 'Q_ref': 0.5, 'openpnm_version': '3.6.1', 'q_ref_raw': 0.5, 'n_inlet_pores': 1, 'n_outlet_pores': 1, 'conductance_model': 'generic_poiseuille', 'solver_voids': 'direct', 'p_ref_min': 0.0, 'p_ref_max': 1.0})
{'k_voids': 0.5, 'k_ref': 0.5, 'Q_voids': 0.5, 'Q_ref': 0.5, 'openpnm_version': '3.6.1', 'q_ref_raw': 0.5, 'n_inlet_pores': 1, 'n_outlet_pores': 1, 'conductance_model': 'generic_poiseuille', 'solver_voids': 'direct', 'p_ref_min': 0.0, 'p_ref_max': 1.0}

Pressure-coupled thermo/OpenPNM cross-check:
SinglePhaseCrosscheckSummary(reference='openpnm_stokesflow', axis='x', permeability_abs_diff=0.0, permeability_rel_diff=0.0, total_flow_abs_diff=0.0, total_flow_rel_diff=0.0, details={'k_voids': 0.5617878367683333, 'k_ref': 0.5617878367683333, 'Q_voids': 56178783.